In [0]:
%pip install azure-servicebus

In [0]:
dbutils.library.restartPython()

In [0]:
import uuid
import random
import json
from datetime import datetime, timedelta
from pyspark.sql.types import *
from pyspark.sql.functions import *

# CONFIGURACAO - Usando DBFS local (storage interno do Databricks)
INPUT_PATH = "/tmp/streaming_input/"
OUTPUT_PATH = "/tmp/streaming_output/"
CHECKPOINT_PATH = "/tmp/checkpoint/"
LOG_PATH = "/tmp/logs/"

# Service Bus (para alertas)
SERVICE_BUS_CONN = "Endpoint=sb://sb-streaming-lab-rafael.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=WP1zoAmOMd0wT5uUpD+jCOOrOA9hSsMik+ASbAirxjk="
QUEUE_NAME = "alertas-fraude"

# Criar diretorios
dbutils.fs.mkdirs(INPUT_PATH)
dbutils.fs.mkdirs(OUTPUT_PATH)
dbutils.fs.mkdirs(CHECKPOINT_PATH)
dbutils.fs.mkdirs(LOG_PATH)

print("✅ Configuracao completa!")
print(f"📁 Input: {INPUT_PATH}")
print(f"📁 Output: {OUTPUT_PATH}")
print(f"📁 Checkpoint: {CHECKPOINT_PATH}")
print(f"📁 Logs: {LOG_PATH}")

In [0]:
# Schema das transacoes
schema_transacoes = StructType([
    StructField("id_transacao", StringType(), True),
    StructField("valor", DoubleType(), True),
    StructField("tipo", StringType(), True),
    StructField("comerciante", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("cpf_cliente", StringType(), True),
    StructField("cidade", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("status", StringType(), True)
])

# Schema de logs
schema_logs = StructType([
    StructField("batch_id", StringType(), True),
    StructField("timestamp_log", TimestampType(), True),
    StructField("arquivo_processado", StringType(), True),
    StructField("total_registros", IntegerType(), True),
    StructField("registros_suspeitos", IntegerType(), True),
    StructField("status_processamento", StringType(), True),
    StructField("mensagem_erro", StringType(), True),
    StructField("tempo_processamento_seg", DoubleType(), True)
])

print("✅ Schemas definidos!")

In [0]:
from azure.servicebus import ServiceBusClient, ServiceBusMessage

def enviar_alerta_fila(mensagem, tipo_alerta="INFO"):
    """Envia mensagem para Service Bus Queue"""
    
    try:
        servicebus_client = ServiceBusClient.from_connection_string(
            conn_str=SERVICE_BUS_CONN
        )
        
        with servicebus_client:
            sender = servicebus_client.get_queue_sender(queue_name=QUEUE_NAME)
            with sender:
                msg_body = {
                    "tipo": tipo_alerta,
                    "mensagem": mensagem,
                    "timestamp": datetime.now().isoformat()
                }
                
                message = ServiceBusMessage(json.dumps(msg_body))
                sender.send_messages(message)
                
        print(f"📤 Alerta enviado: {tipo_alerta} - {mensagem}")
        return True
        
    except Exception as e:
        print(f"❌ Erro ao enviar alerta: {e}")
        return False

print("✅ Funcao de alerta criada!")

In [0]:
import time

def registrar_log(batch_id, arquivo, total_regs, susp_regs, status, erro=None, tempo=0):
    """Registra log de processamento"""
    
    log_data = [{
        "batch_id": str(batch_id),
        "timestamp_log": datetime.now(),
        "arquivo_processado": arquivo,
        "total_registros": int(total_regs),
        "registros_suspeitos": int(susp_regs),
        "status_processamento": status,
        "mensagem_erro": erro if erro else "Sucesso",
        "tempo_processamento_seg": float(tempo)
    }]
    
    df_log = spark.createDataFrame(log_data, schema_logs)
    
    # Salvar no DBFS
    df_log.write.format("delta").mode("append").save(LOG_PATH + "logs")
    
    print(f"📝 Log registrado: Batch {batch_id} - {status}")

print("✅ Funcao de log criada!")

In [0]:
def processar_batch(df_batch, batch_id):
    """Processa micro-batch com deteccao de fraude e alertas"""
    
    inicio = time.time()
    
    try:
        # CORRIGIDO: Usar _metadata.file_path ao invés de input_file_name()
        df_com_fonte = df_batch.withColumn("arquivo_origem", col("_metadata.file_path"))
        
        # Aplicar deteccao de fraude
        df_fraud = df_com_fonte.withColumn(
            "suspeita_fraude",
            when(col("valor") > 5000, "CRITICA")
            .when((col("valor") > 3000) & (col("tipo") == "pix"), "ALTA - PIX")
            .when(col("valor") > 2500, "ALTA")
            .when((col("valor") > 1500) & (col("tipo") == "debito"), "MEDIA")
            .when(col("valor") > 1000, "MEDIA")
            .otherwise("BAIXA")
        )
        
        df_fraud = df_fraud.withColumn(
            "score_risco",
            when(col("suspeita_fraude") == "CRITICA", 95)
            .when(col("suspeita_fraude").contains("ALTA"), 75)
            .when(col("suspeita_fraude").contains("MEDIA"), 45)
            .otherwise(10)
        )
        
        df_fraud = df_fraud.withColumn("timestamp_processamento", current_timestamp())
        df_fraud = df_fraud.withColumn("batch_id_proc", lit(str(batch_id)))
        
        # Coletar metricas
        total_regs = df_fraud.count()
        susp_regs = df_fraud.filter(col("score_risco") >= 45).count()
        criticas = df_fraud.filter(col("score_risco") >= 75).count()
        
        # Salvar dados processados
        df_fraud.write.format("delta").mode("append").save(OUTPUT_PATH + "fraudes")
        
        # Calcular tempo
        tempo_proc = time.time() - inicio
        
        # Registrar log
        arquivo = df_com_fonte.select("arquivo_origem").first()[0] if total_regs > 0 else "vazio"
        registrar_log(batch_id, arquivo, total_regs, susp_regs, "SUCESSO", None, tempo_proc)
        
        # ENVIAR ALERTA se taxa critica alta
        taxa_critica = (criticas / total_regs * 100) if total_regs > 0 else 0
        
        if taxa_critica > 15:
            mensagem = f"Batch {batch_id}: Taxa de fraudes criticas ALTA ({taxa_critica:.2f}%) - {criticas} de {total_regs} transacoes"
            enviar_alerta_fila(mensagem, "ALERTA_CRITICO")
        
        print(f"✅ Batch {batch_id}: {total_regs} trans, {susp_regs} susp, {criticas} crit")
        
    except Exception as e:
        # Em caso de erro
        tempo_proc = time.time() - inicio
        erro_msg = str(e)
        
        registrar_log(batch_id, "erro", 0, 0, "FALHA", erro_msg, tempo_proc)
        
        # Enviar alerta de erro
        mensagem = f"ERRO no Batch {batch_id}: {erro_msg}"
        enviar_alerta_fila(mensagem, "ERRO_PROCESSAMENTO")
        
        print(f"❌ ERRO no Batch {batch_id}: {erro_msg}")
        raise e

print("✅ Funcao de processamento criada (Unity Catalog compativel)!")

In [0]:
print("🚀 Iniciando Stream...")
print(f"📂 Monitorando: {INPUT_PATH}")

# Ler stream
df_stream = spark.readStream \
    .schema(schema_transacoes) \
    .format("json") \
    .option("maxFilesPerTrigger", 1) \
    .load(INPUT_PATH)

# Iniciar stream
query = df_stream.writeStream \
    .foreachBatch(processar_batch) \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(availableNow=True) \
    .start()

print("\n✅ STREAM INICIADO!")
print(f"Query ID: {query.id}")
print("\n📥 Processando arquivos...")
print("\n⏳ Aguardando conclusão...")

query.awaitTermination()

print("\n✅ PROCESSAMENTO CONCLUÍDO!")
enviar_alerta_fila("Processamento concluido - verificar fraudes detectadas", "INFO")

In [0]:
print("🚀 Iniciando Stream...")
print(f"📂 Monitorando: {INPUT_PATH}")

# Ler stream
df_stream = spark.readStream \
    .schema(schema_transacoes) \
    .format("json") \
    .option("maxFilesPerTrigger", 1) \
    .load(INPUT_PATH)

# Iniciar stream com trigger AvailableNow (processa o que está disponível)
query = df_stream.writeStream \
    .foreachBatch(processar_batch) \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(availableNow=True) \
    .start()

print("\n✅ STREAM INICIADO!")
print(f"Query ID: {query.id}")
print("\n📥 Processando arquivos de:", INPUT_PATH)
print("\n⏳ Aguardando conclusão...")

# Aguardar completar
query.awaitTermination()

print("\n✅ PROCESSAMENTO CONCLUÍDO!")

# Enviar notificacao
enviar_alerta_fila("Processamento de streaming concluido", "INFO")

In [0]:
def gerar_cpf():
    """Gera CPF fake"""
    n = [random.randint(0, 9) for _ in range(11)]
    return "{}{}{}.{}{}{}.{}{}{}-{}{}".format(n[0],n[1],n[2],n[3],n[4],n[5],n[6],n[7],n[8],n[9],n[10])

def gerar_arquivo_teste(numero):
    """Gera arquivo JSON de transacoes"""
    
    categorias = ["alimentacao", "transporte", "saude", "lazer", "educacao"]
    tipos = ["debito", "credito", "pix"]
    empresas = ["Padaria", "Supermercado", "Posto", "Uber", "iFood", "Netflix"]
    cidades = ["Sao Paulo", "Rio de Janeiro", "Brasilia", "Curitiba"]
    
    dados = []
    qtd = random.randint(30, 60)
    
    for i in range(qtd):
        sorteio = random.random()
        if sorteio < 0.65:
            v = random.uniform(10, 500)
        elif sorteio < 0.90:
            v = random.uniform(500, 2000)
        elif sorteio < 0.97:
            v = random.uniform(2000, 5000)
        else:
            v = random.uniform(5000, 10000)
        
        valor_final = float(int(v * 100)) / 100.0
        
        transacao = {
            "id_transacao": "TRX_" + str(uuid.uuid4())[:8].upper(),
            "valor": valor_final,
            "tipo": random.choice(tipos),
            "comerciante": random.choice(empresas),
            "timestamp": datetime.now().isoformat(),
            "cpf_cliente": gerar_cpf(),
            "cidade": random.choice(cidades),
            "categoria": random.choice(categorias),
            "status": "processada"
        }
        
        dados.append(transacao)
    
    json_data = "\n".join([json.dumps(d) for d in dados])
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    arquivo_nome = f"transacoes_{timestamp}_{numero}.json"
    arquivo_path = INPUT_PATH + arquivo_nome
    
    dbutils.fs.put(arquivo_path, json_data, True)
    
    print(f"Arquivo {numero}: {arquivo_nome} ({qtd} transacoes)")
    return arquivo_nome

print("Funcao geradora criada!")

In [0]:
print("GERANDO ARQUIVOS DE TESTE")
print("="*60)

for i in range(1, 4):
    arquivo = gerar_arquivo_teste(i)
    print(f"  [{i}/3] {arquivo} -> OK")
    time.sleep(1)

print("\n3 arquivos criados em:", INPUT_PATH)
print("\nAgora execute CELULA 8 para processar!")

In [0]:
print("GERANDO ARQUIVOS DE TESTE")
print("="*60)

for i in range(1, 4):
    arquivo = gerar_arquivo_teste(i)
    print(f"  [{i}/3] {arquivo} -> OK")
    time.sleep(1)

print("\n3 arquivos criados em:", INPUT_PATH)
print("\nAgora execute CELULA 8 para processar!")

In [0]:
# Teste direto de envio para Service Bus

print("TESTE DE CONEXAO COM SERVICE BUS")
print("="*60)

# Verificar se a connection string está configurada
print(f"\nConnection String configurada: {SERVICE_BUS_CONN[:50]}...")
print(f"Queue Name: {QUEUE_NAME}")

# Tentar enviar mensagem de teste
try:
    from azure.servicebus import ServiceBusClient, ServiceBusMessage
    import json
    
    print("\n Conectando ao Service Bus...")
    
    servicebus_client = ServiceBusClient.from_connection_string(
        conn_str=SERVICE_BUS_CONN
    )
    
    with servicebus_client:
        sender = servicebus_client.get_queue_sender(queue_name=QUEUE_NAME)
        
        with sender:
            msg_teste = {
                "tipo": "TESTE",
                "mensagem": "Mensagem de teste do Databricks",
                "timestamp": datetime.now().isoformat()
            }
            
            message = ServiceBusMessage(json.dumps(msg_teste))
            sender.send_messages(message)
            
    print("\n SUCESSO! Mensagem enviada para a fila!")
    print("\nVerifique o Service Bus Explorer no Azure Portal")
    
except Exception as e:
    print(f"\n ERRO ao enviar mensagem!")
    print(f"Erro: {str(e)}")
    print("\nPOSSIVEIS PROBLEMAS:")
    print("1. Connection String incorreta")
    print("2. Nome da fila incorreto")
    print("3. Credenciais sem permissao")